# Benchmark: Spust (Lexicographic K-th Path)

Build C++ binary, generate test instances, time brute-force vs optimised algorithm, plot results.

## 1. Write & Compile C++ Solution

In [6]:
%%writefile spust.cpp
#include <algorithm>
#include <climits>
#include <cstdint>
#include <fstream>
#include <functional>
#include <iostream>
#include <stdexcept>
#include <string>
#include <utility>
#include <vector>

static const std::vector<std::pair<int, int>> deltas = {
    {1, 0}, {-1, 0}, {0, 1}, {0, -1}};
static const char dirs[] = "JSVZ";

struct PathCounter {
    const std::vector<std::vector<int>> &grid;
    std::vector<std::vector<int64_t>> &dp;
    int rows, cols, end_val;

    int64_t count(int r, int c, int64_t budget) {
        if (dp[r][c] != -1)
            return std::min(dp[r][c], budget);
        int64_t total = (grid[r][c] == end_val) ? 1 : 0;
        for (int i = 0; i < 4; i++) {
            if (total >= budget)
                return total;
            int nr = r + deltas[i].first, nc = c + deltas[i].second;
            if (nr >= 0 && nr < rows && nc >= 0 && nc < cols &&
                grid[r][c] > grid[nr][nc])
                total += count(nr, nc, budget - total);
        }
        dp[r][c] = total;
        return total;
    }

    bool trace(int r, int c, int64_t k, int64_t &counted, std::string &path) {
        if (grid[r][c] == end_val) {
            counted++;
            if (counted == k)
                return true;
        }
        for (int i = 0; i < 4; i++) {
            int nr = r + deltas[i].first, nc = c + deltas[i].second;
            if (nr >= 0 && nr < rows && nc >= 0 && nc < cols &&
                grid[r][c] > grid[nr][nc]) {
                int64_t sub = count(nr, nc, k - counted);
                if (counted + sub < k) {
                    counted += sub;
                    continue;
                }
                path.push_back(dirs[i]);
                if (trace(nr, nc, k, counted, path))
                    return true;
            }
        }
        return false;
    }
};

std::string lexicographic_paths(const std::vector<std::vector<int>> &grid,
                                int start_val, int end_val, int64_t k) {
    int rows = grid.size(), cols = grid[0].size();
    int sr = -1, sc = -1;
    for (int r = 0; r < rows; r++)
        for (int c = 0; c < cols; c++)
            if (grid[r][c] == start_val) {
                sr = r;
                sc = c;
            }

    std::vector<std::vector<int64_t>> dp(rows, std::vector<int64_t>(cols, -1));
    PathCounter pc{grid, dp, rows, cols, end_val};

    int64_t counted = 0;
    std::string path;
    path.reserve(rows * cols);
    pc.trace(sr, sc, k, counted, path);
    return path;
}

void brute_force_dfs(const std::vector<std::vector<int>> &grid, int r, int c,
                     int end_val, std::string &current,
                     std::vector<std::string> &all_paths) {
    int rows = grid.size(), cols = grid[0].size();
    if (grid[r][c] == end_val)
        all_paths.push_back(current);
    for (int i = 0; i < 4; i++) {
        int nr = r + deltas[i].first, nc = c + deltas[i].second;
        if (nr >= 0 && nr < rows && nc >= 0 && nc < cols &&
            grid[r][c] > grid[nr][nc]) {
            current.push_back(dirs[i]);
            brute_force_dfs(grid, nr, nc, end_val, current, all_paths);
            current.pop_back();
        }
    }
}

std::string brute_force(const std::vector<std::vector<int>> &grid,
                        int start_val, int end_val, int64_t k) {
    int rows = grid.size(), cols = grid[0].size();

    int sr, sc;
    for (int r = 0; r < rows; r++)
        for (int c = 0; c < cols; c++)
            if (grid[r][c] == start_val) {
                sr = r;
                sc = c;
            }

    std::string cur;
    std::vector<std::string> all_paths;
    brute_force_dfs(grid, sr, sc, end_val, cur, all_paths);
    std::sort(all_paths.begin(), all_paths.end());

    if (k > (int64_t)all_paths.size())
        throw std::runtime_error("Not enough paths");
    return all_paths[k - 1];
}


int main(int argc,char* argv[]){
    if (argc!=3){ std::cerr<<"Usage: spust <input_file> <mode: opt|brute>\n"; return 1; }
    std::ifstream f(argv[1]);
    if (!f){ std::cerr<<"Cannot open: "<<argv[1]<<"\n"; return 1; }
    int64_t N,M,K; f>>N>>M>>K;
    std::vector<std::vector<int>> grid(N,std::vector<int>(M));
    int mn=INT_MAX,mx=INT_MIN;
    for (int i=0;i<N;i++) for (int j=0;j<M;j++){
        f>>grid[i][j]; mn=std::min(mn,grid[i][j]); mx=std::max(mx,grid[i][j]);
    }
    std::string mode=argv[2];
    std::string result = (mode=="brute") ? brute_force(grid,mx,mn,K)
                                         : lexicographic_paths(grid,mx,mn,K);
    std::cout<<result<<"\n";
    return 0;
}


Overwriting spust.cpp


In [7]:
import subprocess, sys

result = subprocess.run(
    ["g++", "-O2", "-std=c++17", "-o", "spust", "spust.cpp"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("COMPILE ERROR:")
    print(result.stderr)
    sys.exit(1)
else:
    print("Compiled OK → ./spust")


Compiled OK → ./spust


## 2. Verify on Problem Example

Expected output: `JZZSZZJJV`

In [8]:
import tempfile, os

example_input = """3 5 3
8 10 11 20 30
7 20 20 21 25
3 1 16 27 20
"""

with open("example.in", "w") as f:
    f.write(example_input)

for mode in ["brute", "opt"]:
    r = subprocess.run(["./spust", "example.in", mode], capture_output=True, text=True)
    print(f"{mode:6s} → {r.stdout.strip()!r}  {'✓' if r.stdout.strip()=='JZZSZZJJV' else '✗ WRONG'}")


brute  → 'JZZSZZJJV'  ✓
opt    → 'JZZSZZJJV'  ✓


## 3. Random Instance Generator

Generates grids with strictly decreasing paths guaranteed (random permutation of 1..N*M).

In [9]:
import numpy as np
import random

def generate_instance(rows, cols, k=1, seed=None):
    """Generate a grid with a guaranteed decreasing path from max to min."""
    grid = np.empty((rows, cols), dtype=int)
    values = np.arange(rows * cols, 0, -1)

    # Lay values on a snake so the solver always has at least one valid path.
    for idx, value in enumerate(values):
        r, offset = divmod(idx, cols)
        c = offset if r % 2 == 0 else cols - 1 - offset
        grid[r, c] = value

    return grid

def write_instance(path, grid, k):
    rows, cols = grid.shape
    with open(path, "w") as f:
        f.write(f"{rows} {cols} {k}\n")
        for row in grid:
            f.write(" ".join(map(str, row)) + "\n")

# Quick visual check
g = generate_instance(3, 5, seed=42)
print("Sample 3x5 grid:")
print(g)
print(f"Max={g.max()}, Min={g.min()}")

Sample 3x5 grid:
[[15 14 13 12 11]
 [ 6  7  8  9 10]
 [ 5  4  3  2  1]]
Max=15, Min=1


## 3. Random Instance Generator

Generates grids with a guaranteed strictly decreasing path from max to min.

In [10]:
errors = 0
checks = 0

for trial in range(50):
    rows = random.randint(2, 4)
    cols = random.randint(2, 4)
    grid = generate_instance(rows, cols, seed=trial)
    write_instance("tmp_check.in", grid, 1)

    r_opt   = subprocess.run(["./spust","tmp_check.in","opt"],   capture_output=True, text=True)
    r_brute = subprocess.run(["./spust","tmp_check.in","brute"], capture_output=True, text=True)

    opt_out   = r_opt.stdout.strip()
    brute_out = r_brute.stdout.strip()

    if r_opt.returncode != 0 or r_brute.returncode != 0:
        continue  # no paths exist, skip

    if opt_out != brute_out:
        print(f"MISMATCH trial={trial} rows={rows} cols={cols}: opt={opt_out!r} brute={brute_out!r}")
        print(grid)
        errors += 1
    checks += 1

print(f"Checked {checks} instances. Errors: {errors}")
if errors == 0:
    print("✓ All match!")


Checked 50 instances. Errors: 0
✓ All match!


## 5. Performance Benchmark

Time both algorithms across increasing grid sizes. Brute-force capped at small sizes (exponential paths).

In [11]:
import time

# Grid sizes to test
sizes_opt   = [5, 6, 7, 8, 9, 10]
sizes_brute = [5, 6, 7, 8, 9, 10]  # brute explodes fast

REPS = 3  # repetitions per size for stable timing

def time_algo(size, mode, reps=REPS, seed=0):
    grid = generate_instance(size, size, seed=seed)
    write_instance("tmp_bench.in", grid, 1)
    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        r = subprocess.run(["./spust", "tmp_bench.in", mode],
                           capture_output=True, text=True)
        t1 = time.perf_counter()
        if r.returncode != 0:
            raise RuntimeError(
                f"spust failed for size={size}, mode={mode}: {r.stderr.strip() or r.stdout.strip()}"
            )
        times.append(t1 - t0)
    return min(times)  # best of reps

print("Benchmarking optimised algorithm...")
times_opt = []
for s in sizes_opt:
    t = time_algo(s, "opt")
    times_opt.append(t)
    print(f"  {s:3d}x{s:3d} → {t*1000:.2f} ms")

print("\nBenchmarking brute-force algorithm...")
times_brute = []
for s in sizes_brute:
    t = time_algo(s, "brute")
    times_brute.append(t)
    print(f"  {s:3d}x{s:3d} → {t*1000:.2f} ms")

Benchmarking optimised algorithm...
    5x  5 → 1.37 ms
    6x  6 → 1.85 ms
    7x  7 → 1.79 ms
    8x  8 → 1.65 ms
    9x  9 → 1.21 ms
   10x 10 → 1.26 ms

Benchmarking brute-force algorithm...
    5x  5 → 1.25 ms
    6x  6 → 1.49 ms
    7x  7 → 4.28 ms
    8x  8 → 43.73 ms
    9x  9 → 531.56 ms
   10x 10 → 9363.70 ms


## 6. Results Plot

In [12]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=[s*s for s in sizes_opt],
    y=[t*1000 for t in times_opt],
    mode="lines+markers",
    name="Optimalen algoritem (DP + iterativni DFS)",
    line=dict(color="royalblue", width=2),
    marker=dict(size=7)
))

fig.add_trace(go.Scatter(
    x=[s*s for s in sizes_brute],
    y=[t*1000 for t in times_brute],
    mode="lines+markers",
    name="Metoda grobe sile (generiraj vse poti)",
    line=dict(color="crimson", width=2, dash="dash"),
    marker=dict(size=7)
))

fig.update_layout(
    title="Primerjava časovne zahtevnosti: optimalen vs. groba sila",
    xaxis_title="Velikost instance (N×M celic)",
    yaxis_title="Čas izvajanja (ms)",
    yaxis_type="log",
    legend=dict(x=0.02, y=0.98),
    template="plotly_white",
    width=900,
    height=500
)

fig.show()


## 7. Bar Chart – Direct Comparison at Common Sizes

In [13]:
common_sizes = [s for s in sizes_brute if s in sizes_opt]
opt_common   = [times_opt[sizes_opt.index(s)]*1000 for s in common_sizes]
brute_common = [times_brute[sizes_brute.index(s)]*1000 for s in common_sizes]

fig2 = go.Figure(data=[
    go.Bar(name="Optimalen algoritem", x=[f"{s}×{s}" for s in common_sizes], y=opt_common,
           marker_color="royalblue"),
    go.Bar(name="Metoda grobe sile",   x=[f"{s}×{s}" for s in common_sizes], y=brute_common,
           marker_color="crimson"),
])

fig2.update_layout(
    barmode="group",
    title="Čas izvajanja po velikosti instance (skupne točke)",
    xaxis_title="Velikost mreže",
    yaxis_title="Čas (ms)",
    yaxis_type="log",
    template="plotly_white",
    width=700,
    height=450
)
fig2.show()


## 8. Speedup Table

In [14]:
import pandas as pd

df = pd.DataFrame({
    "Velikost": [f"{s}×{s}" for s in common_sizes],
    "Optimalen (ms)": [f"{v:.3f}" for v in opt_common],
    "Groba sila (ms)": [f"{v:.3f}" for v in brute_common],
    "Pohitritev (×)": [f"{b/o:.1f}" if o>0 else "N/A"
                       for o,b in zip(opt_common, brute_common)]
})
df


,Velikost,Optimalen (ms),Groba sila (ms),Pohitritev (×)
0,5×5,1.366,1.248,0.9
1,6×6,1.854,1.489,0.8
2,7×7,1.788,4.277,2.4
3,8×8,1.654,43.731,26.4
4,9×9,1.214,531.560,438.0
5,10×10,1.264,9363.695,7409.6


## 9. RAM Usage Test

Measure peak resident memory while the solver is running.

In [15]:
import psutil
import time

def peak_rss_mb(size, mode, seed=0):
    grid = generate_instance(size, size, seed=seed)
    write_instance("tmp_bench.in", grid, 1)

    proc = subprocess.Popen(
        ["./spust", "tmp_bench.in", mode],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    ps_proc = psutil.Process(proc.pid)
    peak_rss = 0

    while proc.poll() is None:
        try:
            peak_rss = max(peak_rss, ps_proc.memory_info().rss)
        except psutil.Error:
            pass
        time.sleep(0.01)

    stdout, stderr = proc.communicate()
    if proc.returncode != 0:
        raise RuntimeError(
            f"spust failed for size={size}, mode={mode}: {stderr.strip() or stdout.strip()}"
        )

    return peak_rss / (1024 * 1024)

ram_sizes_opt = [10, 20, 40, 60, 100]
ram_sizes_brute = [5, 6, 7, 8, 9, 10]

print("RAM usage for optimised algorithm...")
ram_opt = []
for s in ram_sizes_opt:
    rss = peak_rss_mb(s, "opt")
    ram_opt.append(rss)
    print(f"  {s:3d}x{s:3d} → {rss:.2f} MB")

print("\nRAM usage for brute-force algorithm...")
ram_brute = []
for s in ram_sizes_brute:
    rss = peak_rss_mb(s, "brute")
    ram_brute.append(rss)
    print(f"  {s:3d}x{s:3d} → {rss:.2f} MB")


RAM usage for optimised algorithm...
   10x 10 → 0.28 MB
   20x 20 → 0.20 MB
   40x 40 → 0.79 MB
   60x 60 → 0.09 MB
  100x100 → 0.28 MB

RAM usage for brute-force algorithm...
    5x  5 → 0.02 MB
    6x  6 → 0.28 MB
    7x  7 → 0.22 MB
    8x  8 → 18.86 MB
    9x  9 → 201.15 MB
   10x 10 → 4146.57 MB


## 10. RAM Usage Histogram

Show the distribution of measured peak memory values.

In [16]:
import psutil, time, subprocess
import plotly.graph_objects as go

def peak_rss_mb(size, mode, seed=0):
    grid = generate_instance(size, size, seed=seed)
    write_instance("tmp_bench.in", grid, 1)
    proc = subprocess.Popen(
        ["./spust", "tmp_bench.in", mode],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    ps_proc = psutil.Process(proc.pid)
    peak = 0
    while proc.poll() is None:
        try:
            peak = max(peak, ps_proc.memory_info().rss)
        except psutil.Error:
            pass
        time.sleep(0.005)
    proc.wait()
    return peak / (1024 * 1024)

ram_sizes_opt   = [5, 10, 20, 40, 60, 80, 100]
ram_sizes_brute = [5, 6, 7, 8, 9, 10]

print("Measuring RAM — optimised...")
ram_opt = []
for s in ram_sizes_opt:
    v = peak_rss_mb(s, "opt")
    ram_opt.append(v)
    print(f"  {s:3d}x{s:3d} → {v:.2f} MB")

print("\nMeasuring RAM — brute-force...")
ram_brute = []
for s in ram_sizes_brute:
    v = peak_rss_mb(s, "brute")
    ram_brute.append(v)
    print(f"  {s:3d}x{s:3d} → {v:.2f} MB")

# ── Line chart ──────────────────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=[s*s for s in ram_sizes_opt],
    y=ram_opt,
    mode="lines+markers",
    name="Optimalen algoritem",
    line=dict(color="royalblue", width=2),
    marker=dict(size=8, symbol="circle"),
))

fig.add_trace(go.Scatter(
    x=[s*s for s in ram_sizes_brute],
    y=ram_brute,
    mode="lines+markers",
    name="Metoda grobe sile",
    line=dict(color="crimson", width=2, dash="dash"),
    marker=dict(size=8, symbol="diamond"),
))

fig.update_layout(
    title="Poraba pomnilnika (peak RSS) glede na velikost instance",
    xaxis_title="Velikost instance (N×M celic)",
    yaxis_title="Peak RSS (MB)",
    legend=dict(x=0.02, y=0.98),
    template="plotly_white",
    width=900,
    height=500,
)

fig.show()


Measuring RAM — optimised...
    5x  5 → 0.27 MB
   10x 10 → 0.21 MB
   20x 20 → 0.28 MB
   40x 40 → 0.09 MB
   60x 60 → 0.28 MB
   80x 80 → 0.02 MB
  100x100 → 0.08 MB

Measuring RAM — brute-force...
    5x  5 → 0.28 MB
    6x  6 → 0.43 MB
    7x  7 → 0.09 MB
    8x  8 → 17.84 MB
    9x  9 → 228.15 MB
   10x 10 → 4140.17 MB


In [ ]:
import pandas as pd

# Align on common sizes
common_ram = [s for s in ram_sizes_brute if s in ram_sizes_opt]
opt_ram_c   = [ram_opt[ram_sizes_opt.index(s)]   for s in common_ram]
brute_ram_c = [ram_brute[ram_sizes_brute.index(s)] for s in common_ram]

pd.DataFrame({
    "Velikost": [f"{s}×{s}" for s in common_ram],
    "Optimalen (MB)":  [f"{v:.2f}" for v in opt_ram_c],
    "Groba sila (MB)": [f"{v:.2f}" for v in brute_ram_c],
    "Razmerje (×)": [f"{b/o:.2f}" if o>0 else "N/A"
                     for o,b in zip(opt_ram_c, brute_ram_c)],
})


,Velikost,Optimalen (MB),Groba sila (MB),Razmerje (×)
0,5×5,0.07,0.07,1.00
1,10×10,0.28,4143.15,14731.21
